# Day 1: Data Quality Introduction and Profiling

This notebook introduces practical **data quality assessment**.

Learning objectives:
1. detect common issues,
2. profile a dataset from a quality perspective,
3. map findings to quality dimensions,
4. explain business impact.

> Week 2 focus: detect, measure, validate, and report (not full cleaning pipelines).


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
repo_root = Path.cwd().resolve().parents[1]
data_path = repo_root / 'data' / 'raw' / 'week2_healthcare_visits_messy.csv'
df = pd.read_csv(data_path)
print('Shape:', df.shape)
df.head()


## Step 1 - First profile

Start with structure, types, and basic summary.


In [ ]:
df.info()

df.describe(include='all').transpose()


## Step 2 - Missing values (Completeness)


In [ ]:
missing = pd.DataFrame({'missing_count':df.isna().sum(),'missing_pct':(df.isna().mean()*100).round(1)}).sort_values('missing_pct',ascending=False)
missing


In [ ]:
missing['missing_pct'].plot(kind='bar', figsize=(10,4), title='Missing values by column (%)')
plt.tight_layout(); plt.show()


## Step 3 - Duplicates (Uniqueness)


In [ ]:
print('Full-row duplicates:', df.duplicated().sum())
print('Duplicate visit_id:', df.duplicated(subset=['visit_id']).sum())
df[df.duplicated(subset=['visit_id'], keep=False)].sort_values('visit_id')


## Step 4 - Invalid values (Validity and Accuracy)


In [ ]:
age = pd.to_numeric(df['age'], errors='coerce')
hr = pd.to_numeric(df['heart_rate'], errors='coerce')
invalid_age = (age<0) | (age>120) | age.isna()
invalid_hr = (hr<30) | (hr>220) | hr.isna()
invalid_gender = ~df['gender'].isin(['M','F','Male','Female']) & df['gender'].notna()
print('Invalid age rows:', int(invalid_age.sum()))
print('Invalid heart_rate rows:', int(invalid_hr.sum()))
print('Invalid gender rows:', int(invalid_gender.sum()))
df[invalid_age | invalid_hr | invalid_gender][['visit_id','age','heart_rate','gender']]


## Step 5 - Inconsistent formats (Consistency)


In [ ]:
visit_dt = pd.to_datetime(df['visit_date'], errors='coerce', format='mixed')
upd_dt = pd.to_datetime(df['last_updated'], errors='coerce', format='mixed')
fmt_issues = df[visit_dt.isna() | upd_dt.isna()][['visit_id','visit_date','last_updated']]
fmt_issues


## Step 6 - Simple integrity checks (Integrity and Timeliness)


In [ ]:
missing_keys = df['visit_id'].isna() | df['patient_id'].isna() | (df['visit_id'].astype(str).str.strip()=='')
update_before_visit = upd_dt < visit_dt
valid_patients = {'P1001','P1002','P1004','P1005','P1006','P1007','P1008','P1010'}
unknown_pid = df['patient_id'].notna() & ~df['patient_id'].isin(valid_patients)
print('Missing key fields:', int(missing_keys.sum()))
print('last_updated < visit_date:', int(update_before_visit.sum()))
print('Unknown patient IDs:', int(unknown_pid.sum()))


## Step 7 - Map findings to dimensions


In [ ]:
dimension_map = pd.DataFrame([
('Accuracy','Out-of-range numeric values indicate likely entry errors'),
('Completeness','Missing IDs and critical fields'),
('Consistency','Mixed labels and date formats across records'),
('Timeliness','Some records updated too late or invalid update dates'),
('Uniqueness','Duplicate records and duplicate IDs'),
('Integrity','Unknown patient references and key issues'),
('Validity','Values violating accepted rules or ranges'),
('Relevance','Some fields may be less useful for current KPI goals'),
('Traceability (Lineage)','No source-system column to track issue origin'),
], columns=['Dimension','Observed issue'])
dimension_map


## Function spotlight - profiling helpers

To make the code easier to reuse, we wrap repeated logic into small functions.

### Why this helps
- students can understand *inputs* and *outputs* clearly,
- parameters make checks configurable,
- the same logic can be reused in later audits.


In [ ]:
def summarize_missing_values(dataframe, sort_desc=True):
    """Return missing-value count and percentage per column.

    Parameters
    ----------
    dataframe : pd.DataFrame
        The dataset to inspect.
    sort_desc : bool, default=True
        If True, sort columns from highest to lowest missing percentage.

    Returns
    -------
    pd.DataFrame
        Table with 'missing_count' and 'missing_pct'.
    """
    result = pd.DataFrame({
        'missing_count': dataframe.isna().sum(),
        'missing_pct': (dataframe.isna().mean() * 100).round(1),
    })
    return result.sort_values('missing_pct', ascending=not sort_desc)


def count_duplicates(dataframe, key_columns=None):
    """Count duplicates either on full row or selected key columns.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Input dataset.
    key_columns : list[str] or None, default=None
        - None -> check exact full-row duplicates.
        - list -> check duplicates based on business keys (e.g., ['visit_id']).

    Returns
    -------
    int
        Number of duplicated rows.
    """
    return int(dataframe.duplicated(subset=key_columns).sum())


missing_via_fn = summarize_missing_values(df)
print('Missing-value summary built with function:')
display(missing_via_fn.head())

print('Full-row duplicates (function):', count_duplicates(df))
print('visit_id duplicates (function):', count_duplicates(df, key_columns=['visit_id']))


## Reflective questions

1. Which issue would you prioritize first?
2. Which issue most affects decision quality?
3. What evidence is needed before cleaning?


## What did we learn?

Good quality work starts with **evidence-based assessment**.
Week 2 builds the ability to detect and report issues before implementing cleaning pipelines in the next chapter.
